# MS1 Window-level Contrastive Learning (Minimal Preprocessing Aligned)

This notebook is adapted for the **minimal preprocessing** dataset:

- uses `rt_grid` / `signal_grid`
- keeps `window_size=500`, `stride=250`, `jitter_max=10` in **points**
- uses **unsupervised** file-level positives for SupCon
- uses lighter augmentation for the minimal-input setting
- adds stronger reproducibility controls (seeded RNG in dataset / augment / slicer)


In [ ]:
# --- Imports + core classes (minimal-preprocessing aligned) ---


import os, re
from pathlib import Path
import math
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

# -------------------------
# Global config
# -------------------------
SEED = 0
# META_PATH = "data/processed/metadata_with_frog.csv"
#NPZ_DIR = "data/processed/mzML_npz_45"
#LOG_ROOT = Path("results/runs/training/Task1710")


PROJECT_ROOT = Path("../..").resolve()
META_PATH=PROJECT_ROOT / "data" / "processed" / "metadata_with_frog.csv"
NPZ_DIR=PROJECT_ROOT / "data" / "processed" /"mzML_npz_45"
LOG_ROOT=PROJECT_ROOT /"results"/"runs"/"training"/ "temperature_sensitivity" / "Task1710_tau0p1"
os.makedirs(LOG_ROOT, exist_ok=True)






def seed_everything_all(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    L.seed_everything(seed, workers=True)

seed_everything_all(SEED)

meta = pd.read_csv(META_PATH, dtype=str)
uhm2trt = dict(zip(meta["UHM_sample"], meta["treatment"]))

def normalize_uhm_from_dataset_filename(fn: str) -> str:
    s = os.path.basename(str(fn))
    s = re.sub(r"\.(npz|mzML)$", "", s)
    s = re.sub(r"_MS1$", "", s)
    return s

def file_name_to_treatment(file_name: str) -> str:
    uhm = normalize_uhm_from_dataset_filename(file_name)
    trt = uhm2trt.get(uhm, None)
    if trt is None:
        raise KeyError(f"UHM '{uhm}' not found in metadata. file_name={file_name}")
    return trt


class ChromAugment:
    """
    Conservative LC-MS augmentation for the minimal-preprocessing data.
    Signal only; RT is kept unchanged.

    Notes
    -----
    - Uses its own RNG for reproducibility.
    - p95 is used as a robust intensity reference.
    """
    def __init__(
        self,
        amp_scale_range=(0.9, 1.1),
        baseline_offset_frac=0.0,
        noise_frac=0.005,
        clip_min=0.0,
        seed: int = 0,
    ):
        self.amp_scale_range = tuple(amp_scale_range)
        self.baseline_offset_frac = float(baseline_offset_frac)
        self.noise_frac = float(noise_frac)
        self.clip_min = clip_min
        self.rng = np.random.default_rng(seed)

    def __call__(self, chromatogram: dict) -> dict:
        rt = chromatogram["rt"]
        sig = chromatogram["signal"]

        if hasattr(rt, "cpu"):
            rt = rt.cpu().numpy()
        if hasattr(sig, "cpu"):
            sig = sig.cpu().numpy()

        rt = np.asarray(rt, dtype=np.float32)
        sig = np.asarray(sig, dtype=np.float32)

        p95 = float(np.percentile(np.abs(sig), 95)) + 1e-12

        a = float(self.rng.uniform(*self.amp_scale_range))
        sig2 = sig * a

        if self.baseline_offset_frac > 0:
            b = float(self.rng.uniform(-self.baseline_offset_frac, self.baseline_offset_frac)) * p95
            sig2 = sig2 + b

        if self.noise_frac > 0:
            noise_std = self.noise_frac * p95
            sig2 = sig2 + self.rng.normal(0.0, noise_std, size=sig2.shape).astype(np.float32)

        if self.clip_min is not None:
            sig2 = np.maximum(sig2, self.clip_min).astype(np.float32)

        out = dict(chromatogram)
        out["rt"] = rt.astype(np.float32)
        out["signal"] = sig2.astype(np.float32)
        return out


class WindowSlicer:
    """
    Window slicing in POINTS (not seconds).

    Current convention:
    - window_size=500 points
    - stride=250 points
    - jitter_max=10 points

    Since the minimal dataset is already on a shared RT grid (~0.2 sec / point),
    these point-based settings are now stable across files.
    """
    def __init__(self, window_size=500, stride=250, jitter_max=0, seed: int = 0):
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.jitter_max = int(jitter_max)
        self.rng = np.random.default_rng(seed)

    def __call__(self, chromatogram):
        rt = chromatogram["rt"]
        signal = chromatogram["signal"]
        chrom_id = chromatogram.get("chrom_id", None)
        chrom_name = chromatogram.get("chrom_name", None)

        if chrom_id is None:
            raise KeyError("chromatogram must contain 'chrom_id'")

        if hasattr(rt, "cpu"):
            rt = rt.cpu().numpy()
        if hasattr(signal, "cpu"):
            signal = signal.cpu().numpy()

        rt = np.asarray(rt)
        signal = np.asarray(signal)

        Lsig = len(rt)
        windows = []
        start = 0

        j = int(self.rng.integers(-self.jitter_max, self.jitter_max + 1)) if self.jitter_max > 0 else 0

        while start + self.window_size <= Lsig:
            s = start + j
            s = max(0, min(s, Lsig - self.window_size))
            e = s + self.window_size

            windows.append({
                "rt": rt[s:e].astype(np.float32),
                "signal": signal[s:e].astype(np.float32),
                "start": int(s),
                "chrom_id": int(chrom_id),
                "chrom_name": chrom_name,
                "jitter": int(j),
            })
            start += self.stride

        return windows


class MassSpecWindowDataset(torch.utils.data.Dataset):
    """
    Window-level dataset.
    Each MS1 file:
      -> two independent file-level augmentations
      -> slice into windows AFTER augmentation
    """
    def __init__(
        self,
        npz_files,
        window_slicer,
        augment: ChromAugment | None = None,
        n_windows_target: int = 12,
        seed: int = 0,
    ):
        self.npz_files = [Path(p) for p in npz_files]
        self.slicer = window_slicer
        self.augment = augment
        self.n_windows_target = n_windows_target
        self.rng = np.random.default_rng(seed)
        self.file_id_map = {p.name: i for i, p in enumerate(self.npz_files)}

    def __len__(self):
        return len(self.npz_files)

    def _load_chrom(self, npz_path: Path) -> dict:
        d = np.load(npz_path)
        file_id = self.file_id_map[npz_path.name]
        return {
            "rt": d["rt_grid"].astype(np.float32),
            "signal": d["signal_grid"].astype(np.float32),
            "chrom_id": int(file_id),
            "chrom_name": npz_path.name,
        }

    def _sample_windows(self, windows: list[dict]) -> list[dict]:
        n = len(windows)
        k = self.n_windows_target
        if k is None:
            return windows
        if n == 0:
            return []

        if n >= k:
            idx = self.rng.choice(n, size=k, replace=False)
        else:
            idx = self.rng.choice(n, size=k, replace=True)

        return [windows[i] for i in idx]

    def __getitem__(self, idx):
        npz_path = self.npz_files[idx]
        chrom = self._load_chrom(npz_path)

        chromA = self.augment(chrom) if self.augment is not None else chrom
        chromB = self.augment(chrom) if self.augment is not None else chrom

        assert isinstance(chromA, dict) and isinstance(chromB, dict), (type(chromA), type(chromB))

        winsA = self._sample_windows(self.slicer(chromA))
        winsB = self._sample_windows(self.slicer(chromB))

        def pack(wins):
            X = []
            for w in wins:
                sig = w["signal"].astype(np.float32)
                rt = w["rt"].astype(np.float32)
                X.append(np.stack([sig, rt], axis=1))
            return np.stack(X, axis=0)  # (N,L,2)

        XA = pack(winsA)
        XB = pack(winsB)
        chrom_id = chrom["chrom_id"]
        chrom_name = chrom["chrom_name"]
        return {
            "x": np.stack([XA, XB], axis=0),  # (2,N,L,2)
            "file_id": int(chrom_id),
            "file_name": chrom_name,
            "chrom_id": int(chrom_id),
            "chrom_name": chrom_name,
        }


class MassSpecWindowContrastEncoder(nn.Module):
    """
    Window-level encoder:
    Patch(Conv1d) + Transformer + CLS + RT sinusoidal positional encoding.
    Input:  signal(B,L), rt(B,L)
    Output: embedding(B,embed_dim)
    """
    def __init__(self, patch_size=50, stride=50, embed_dim=64, num_heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.stride = stride
        self.embed_dim = embed_dim

        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride,
            padding=0,
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.positional_encoding = SinusoidalPositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, signal, rt):
        B = signal.size(0)
    
        if not torch.isfinite(signal).all():
            raise RuntimeError("signal contains NaN/Inf before encoder")
        if not torch.isfinite(rt).all():
            raise RuntimeError("rt contains NaN/Inf before encoder")
    
        # light numeric stabilization: keep shape, only shrink magnitude
        signal = signal / 1e6
    
        x = self.conv(signal.unsqueeze(1))   # (B, embed_dim, n_patches)
        if not torch.isfinite(x).all():
            raise RuntimeError("conv output contains NaN/Inf")
    
        x = x.permute(0, 2, 1)               # (B, n_patches, embed_dim)
    
        rt_patch = rt[:, self.patch_size - 1::self.stride]
        assert x.size(1) == rt_patch.size(1), (
            f"Patch/RT mismatch: conv patches={x.size(1)} vs rt patches={rt_patch.size(1)}. "
            f"patch_size={self.patch_size}, stride={self.stride}, input_len={rt.size(1)}"
        )
    
        rt_patch = rt_patch / 1800.0
        pe = self.positional_encoding(rt_patch)
        if not torch.isfinite(pe).all():
            raise RuntimeError("positional encoding contains NaN/Inf")
    
        x = x + pe
        if not torch.isfinite(x).all():
            raise RuntimeError("x + positional encoding contains NaN/Inf")
    
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
    
        x = self.transformer(x)
        if not torch.isfinite(x).all():
            raise RuntimeError("transformer output contains NaN/Inf")
    
        h = x[:, 0, :]
        #h = x[:, 1:, :].mean(dim=1)
        h = F.layer_norm(h, h.shape[-1:])
        if not torch.isfinite(h).all():
            raise RuntimeError("encoder output h contains NaN/Inf")
    
        return h


class SupConLoss_mean(nn.Module):
    """Stable SupCon (mean over positives) using logsumexp."""
    def __init__(self, temperature=0.5, eps: float = 1e-8):
        super().__init__()
        self.temperature = temperature
        self.eps = float(eps)

    def forward(self, features, labels):
        device = features.device
        N = features.size(0)

        z = F.normalize(features, dim=1, eps=self.eps)
        labels = labels.view(-1, 1).to(device)

        pos_mask = torch.eq(labels, labels.T)
        self_mask = torch.eye(N, device=device, dtype=torch.bool)
        pos_mask = pos_mask & (~self_mask)

        logits = (z @ z.T) / self.temperature

        # 数值稳定：每行减去最大值
        logits_max, _ = logits.max(dim=1, keepdim=True)
        logits = logits - logits_max.detach()

        # 去掉 self
        logits = logits.masked_fill(self_mask, float("-inf"))

        log_denom = torch.logsumexp(logits, dim=1, keepdim=True)
        log_prob = logits - log_denom

        # 只在 positive 位置取值，别让无关位置的 nan/inf 混进来
        pos_count = pos_mask.sum(dim=1).float()
        valid = pos_count > 0

        if not valid.any():
            return torch.zeros([], device=device, dtype=features.dtype, requires_grad=True)

        log_prob_pos = log_prob.masked_fill(~pos_mask, 0.0)
        mean_log_prob_pos = log_prob_pos.sum(dim=1) / (pos_count + self.eps)

        loss = -mean_log_prob_pos[valid].mean()

        if not torch.isfinite(loss):
            raise RuntimeError("SupCon loss became NaN/Inf")

        return loss


# class ProjectionHead(nn.Module):
#     def __init__(self, in_dim=64, hidden_dim=128, out_dim=64, use_bn=False):
#         super().__init__()
#         layers = [nn.Linear(in_dim, hidden_dim)]
#         if use_bn:
#             layers.append(nn.BatchNorm1d(hidden_dim))
#         layers += [nn.ReLU(inplace=True), nn.Linear(hidden_dim, out_dim)]
#         self.net = nn.Sequential(*layers)

#     def forward(self, x):
#         return self.net(x)


class ProjectionHead(nn.Module):
    def __init__(self, in_dim=64, hidden_dim=128, out_dim=64,use_bn=False):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),   # 👈 替换 BN
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

        

class WindowEncoderWithHead(nn.Module):
    """encoder -> h ; projection head -> z"""
    def __init__(self, encoder, proj_hidden=128, proj_out=64, use_bn=False):
        super().__init__()
        self.encoder = encoder

        in_dim = getattr(encoder, "embed_dim", None)
        if in_dim is None:
            raise ValueError("encoder must have attribute `embed_dim`")

        self.proj = ProjectionHead(
            in_dim=in_dim,
            hidden_dim=proj_hidden,
            out_dim=proj_out,
            use_bn=use_bn,
        )

    def forward(self, signal, rt):
        h = self.encoder(signal, rt)
        if not torch.isfinite(h).all():
            raise RuntimeError("h contains NaN/Inf in WindowEncoderWithHead")
    
        z = self.proj(h)
        if not torch.isfinite(z).all():
            raise RuntimeError("z contains NaN/Inf in WindowEncoderWithHead")
    
        return h, z


class MassSpecWindowContrast(L.LightningModule):
    """Window-level contrastive learning."""
    def __init__(
        self,
        encoder: nn.Module,
        temperature: float = 0.5,
        lr: float = 1e-3,
        weight_decay: float = 1e-4,
        log_every_n_steps: int = 10,
    ):
        super().__init__()
        self.encoder = encoder
        self.temperature = temperature
        self.lr = lr
        self.weight_decay = weight_decay
        self.log_every_n_steps = log_every_n_steps
        self.enc_with_head = WindowEncoderWithHead(self.encoder, proj_hidden=128, proj_out=64, use_bn=False)
        self.supcon_loss = SupConLoss_mean(temperature=temperature)
        self.save_hyperparameters(ignore=["encoder"])

    def training_step(self, batch, batch_idx):
        signal = batch["signal"].to(self.device)
        rt = batch["rt"].to(self.device)
        y = batch["y_file"].to(self.device)
    
        h, z = self.enc_with_head(signal, rt)
        loss = self.supcon_loss(z, y)
        self.log("train/loss", loss, prog_bar=True)
    
        with torch.no_grad():
            feat = F.normalize(z, dim=1)
            N = feat.size(0)
    
            std_mean = feat.std(dim=0).mean()
            self.log("debug/std_mean", std_mean, prog_bar=True)
    
            m = min(512, N)
            idx = torch.randperm(N, device=feat.device)[:m]
            feat_sub = feat[idx]
            lbl = y[idx]
    
            sim = feat_sub @ feat_sub.T
            mask_offdiag = ~torch.eye(m, dtype=torch.bool, device=feat.device)
            sim_off = sim[mask_offdiag]
    
            if sim_off.numel() > 0:
                self.log("debug/sim_off_mean", sim_off.mean(), prog_bar=True)
                self.log("debug/sim_off_max", sim_off.max(), prog_bar=True)
    
            sim_for_nn = sim.masked_fill(~mask_offdiag, -1e9)
            nn_idx = sim_for_nn.argmax(dim=1)
            nn_acc = (lbl[nn_idx] == lbl).float().mean()
            self.log("train/nn_top1", nn_acc, prog_bar=True)
    
        with torch.no_grad():
            feat_h = F.normalize(h, dim=1)
            std_h = feat_h.std(dim=0).mean()
            self.log("debug/h_std_mean", std_h)
    
            feat_h_sub = feat_h[idx]
            sim_h = feat_h_sub @ feat_h_sub.T
            mask_h_offdiag = ~torch.eye(m, dtype=torch.bool, device=feat_h.device)
            sim_h_off = sim_h[mask_h_offdiag]
    
            if sim_h_off.numel() > 0:
                self.log("debug/h_sim_off_mean", sim_h_off.mean())
    
        return loss
    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.register_buffer("div_term", div_term)

    def forward(self, rt):
        rt = rt.unsqueeze(-1)  # (B,seq,1)
        pe = torch.zeros(rt.size(0), rt.size(1), self.d_model, device=rt.device)
        pe[:, :, 0::2] = torch.sin(rt * self.div_term)
        pe[:, :, 1::2] = torch.cos(rt * self.div_term)
        return pe


def collate_window_level_with_name(batch):
    """
    Used for:
    - training (SupCon): y_file
    - evaluation (PCA / LDA): file_name
    """
    sigs, rts, y_files, names = [], [], [], []

    for b in batch:
        x = b["x"]   # (2,N,L,2)
        V, N, Lsig, C = x.shape

        signal = x[..., 0].reshape(V * N, Lsig)
        rt = x[..., 1].reshape(V * N, Lsig)

        sigs.append(torch.from_numpy(signal))
        rts.append(torch.from_numpy(rt))

        fid = b.get("file_id", b.get("chrom_id"))
        if fid is not None:
            y = np.full((V * N,), int(fid), dtype=np.int64)
            y_files.append(torch.from_numpy(y))

        name = b.get("file_name", b.get("chrom_name"))
        names.extend([name] * (V * N))

    out = {
        "signal": torch.cat(sigs, dim=0).float(),
        "rt": torch.cat(rts, dim=0).float(),
        "file_name": names,
    }

    if len(y_files) > 0:
        out["y_file"] = torch.cat(y_files, dim=0).long()

    return out




# =========================
# 0) imports
# =========================
from pathlib import Path
import json
import itertools
import pandas as pd
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader


# =========================
# 1) single run
# =========================
def run_phase1_train(
    seed,
    exp_name,
    npz_dir,
    log_root,
    window_size=500,
    stride=250,
    jitter_max=10,
    amp_scale_range=(0.9, 1.1),
    baseline_offset_frac=0.0,
    noise_frac=0.005,
    n_windows_target=32,
    batch_size=32,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
    temperature=1.0,
    lr=1e-4,
    max_epochs=300,
    every_n_train_steps=50,
    gradient_clip_val=1.0,
    devices=1,
):
    seed_everything_all(seed)

    npz_dir = Path(npz_dir)
    log_root = Path(log_root)
    log_root.mkdir(parents=True, exist_ok=True)

    npz_files = sorted(npz_dir.glob("*.npz"))
    print("=" * 80)
    print("Experiment:", exp_name)
    print("Seed:", seed)
    print("Total npz:", len(npz_files))
    print("NPZ_DIR:", npz_dir)

    slicer = WindowSlicer(
        window_size=window_size,
        stride=stride,
        jitter_max=jitter_max,
        seed=seed,
    )

    augment = ChromAugment(
        amp_scale_range=amp_scale_range,
        baseline_offset_frac=baseline_offset_frac,
        noise_frac=noise_frac,
        clip_min=0.0,
        seed=seed,
    )

    dataset = MassSpecWindowDataset(
        npz_files=npz_files,
        window_slicer=slicer,
        augment=augment,
        n_windows_target=n_windows_target,
        seed=seed,
    )

    train_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_window_level_with_name,
        drop_last=False,
    )

    model = MassSpecWindowContrast(
        encoder=MassSpecWindowContrastEncoder(
            embed_dim=embed_dim,
            patch_size=patch_size,
            stride=encoder_stride,
            num_heads=num_heads,
        ),
        temperature=temperature,
        lr=lr,
    )

    logger = CSVLogger(save_dir=str(log_root), name=exp_name)

    checkpoint_cb = ModelCheckpoint(
        dirpath=None,
        save_top_k=-1,
        every_n_train_steps=every_n_train_steps,
        filename="step{step}-loss{train/loss:.3f}",
        auto_insert_metric_name=False,
    )

    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=devices,
        gradient_clip_val=gradient_clip_val,
        logger=logger,
        callbacks=[checkpoint_cb],
        log_every_n_steps=1,
    )

    cfg = {
        "seed": seed,
        "exp_name": exp_name,
        "npz_dir": str(npz_dir),
        "log_root": str(log_root),
        "window_size": window_size,
        "stride": stride,
        "jitter_max": jitter_max,
        "amp_scale_range": amp_scale_range,
        "baseline_offset_frac": baseline_offset_frac,
        "noise_frac": noise_frac,
        "n_windows_target": n_windows_target,
        "batch_size": batch_size,
        "embed_dim": embed_dim,
        "patch_size": patch_size,
        "encoder_stride": encoder_stride,
        "num_heads": num_heads,
        "temperature": temperature,
        "lr": lr,
        "max_epochs": max_epochs,
        "every_n_train_steps": every_n_train_steps,
        "gradient_clip_val": gradient_clip_val,
        "devices": devices,
    }

    trainer.fit(model, train_loader)

    ckpt_dir = Path(checkpoint_cb.dirpath)
    print("Saved checkpoints in:", ckpt_dir)

    cfg_path = ckpt_dir.parent / "run_config.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    out = {
        "exp_name": exp_name,
        "seed": seed,
        "ckpt_dir": str(ckpt_dir),
        "config_path": str(cfg_path),
    }
    out.update(cfg)
    return out


# =========================
# 2) same config, multiple seeds
# =========================
def run_phase1_multi_seed(
    exp_base_name,
    seeds,
    npz_dir,
    log_root,
    **train_kwargs,
):
    rows = []

    for seed in seeds:
        exp_name = f"{exp_base_name}_seed{seed}"
        out = run_phase1_train(
            seed=seed,
            exp_name=exp_name,
            npz_dir=npz_dir,
            log_root=log_root,
            **train_kwargs,
        )
        rows.append(out)

    return pd.DataFrame(rows)


# =========================
# 3) grid helper
# =========================
def expand_param_grid(param_grid: dict):
    keys = list(param_grid.keys())
    values = [param_grid[k] if isinstance(param_grid[k], (list, tuple)) else [param_grid[k]] for k in keys]
    rows = []
    for combo in itertools.product(*values):
        rows.append(dict(zip(keys, combo)))
    return rows


def make_config_name(base_name: str, cfg: dict):
    parts = [base_name]
    for k, v in cfg.items():
        if isinstance(v, tuple):
            v = "-".join(map(str, v))
        s = str(v).replace(" ", "").replace("/", "_")
        parts.append(f"{k}-{s}")
    return "__".join(parts)


def run_phase1_grid(
    exp_base_name,
    seeds,
    npz_dir,
    log_root,
    param_grid,
    common_kwargs=None,
):
    common_kwargs = {} if common_kwargs is None else dict(common_kwargs)
    grid = expand_param_grid(param_grid)

    all_rows = []
    print(f"Total grid configs: {len(grid)}")

    for i, cfg in enumerate(grid, start=1):
        cfg_name = make_config_name(exp_base_name, cfg)
        print("\n" + "#" * 100)
        print(f"[GRID {i}/{len(grid)}] {cfg_name}")
        print(json.dumps(cfg, indent=2))

        df_runs = run_phase1_multi_seed(
            exp_base_name=cfg_name,
            seeds=seeds,
            npz_dir=npz_dir,
            log_root=log_root,
            **common_kwargs,
            **cfg,
        )
        df_runs["config"] = cfg_name
        df_runs["grid_params"] = [json.dumps(cfg, sort_keys=True)] * len(df_runs)
        all_rows.append(df_runs)

    if len(all_rows) == 0:
        return pd.DataFrame()

    return pd.concat(all_rows, ignore_index=True)


In [ ]:

# Example: contrastive-learning hyperparameter grid
COMMON_KWARGS = dict(
    window_size=500,
    stride=250,
    n_windows_target=32,
    batch_size=32,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
    max_epochs=400,
    every_n_train_steps=50,
    gradient_clip_val=1.0,
    devices=1,
)

PARAM_GRID = {
    "jitter_max": [10],
    "amp_scale_range": [(0.99,1.01)],
    "noise_frac": [0.003],
    "temperature": [0.1],
    "lr": [1e-4],
}

df_runs_grid = run_phase1_grid(
    exp_base_name="phase1_ln_grid",
    seeds=[0,1,2,3,4],
    npz_dir=NPZ_DIR,
    log_root=LOG_ROOT,
    param_grid=PARAM_GRID,
    common_kwargs=COMMON_KWARGS,
)

display(df_runs_grid.head())
print("Total runs:", len(df_runs_grid))


## Evaluate all saved checkpoints (task-wise, no loss filtering, no auto-selection)

This section:

- evaluates **every saved checkpoint** for **both downstream tasks**
- records `config`, `seed`, `ckpt_path`, `step`
- records frog-level `AUROC`, `AUPRC`, `ACC` for each task
- computes helper columns:
  - `mean_AUROC_ctrl`
  - `mean_AUPRC_ctrl`
  - `mean_ACC_ctrl`

It does **not** choose the final best checkpoint or final best config automatically.

In [ ]:

# =========================
# EVAL: imports
# =========================
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
from torch.utils.data import DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score


# =========================
# A) step / exp / seed / config parser
# =========================
def parse_step_from_ckpt_name(s):
    m = re.search(r"step(\d+)-loss", str(s))
    return int(m.group(1)) if m else None


def parse_exp_name_from_ckpt_path(ckpt_path):
    p = Path(ckpt_path)
    parts = p.parts
    if "checkpoints" in parts:
        i = parts.index("checkpoints")
        if i >= 2:
            return parts[i - 2]
    return None


def parse_seed_from_exp_name(exp_name):
    if exp_name is None:
        return None
    m = re.search(r"_seed(\d+)$", str(exp_name))
    return int(m.group(1)) if m else None


def parse_config_from_exp_name(exp_name):
    if exp_name is None:
        return None
    return re.sub(r"_seed\d+$", "", str(exp_name))


# =========================
# B) metadata helpers
# =========================
def strip_prefix_any(sd, prefixes):
    for p in prefixes:
        if any(k.startswith(p) for k in sd.keys()):
            out = {k[len(p):]: v for k, v in sd.items() if k.startswith(p)}
            return out, p
    return None, None


def frog_from_uhm_sample(s: str) -> str:
    return str(s).split("-")[0]


def frog_from_npz_name(fn: str) -> str:
    return str(fn).split("-")[0]


def sample_id_from_npz_name(npz_name: str) -> str:
    base = Path(npz_name).name
    m = re.search(r"-(\d+)", base)
    return m.group(1) if m else base


def sample_id_from_meta_filename(meta_filename: str) -> str:
    base = str(meta_filename)
    m = re.search(r"WF22_(\d+)", base)
    return m.group(1) if m else base


def build_npz_to_label_maps(npz_files, meta_csv):
    df = pd.read_csv(meta_csv)
    assert "UHM_sample" in df.columns
    assert "treatment" in df.columns
    assert "filename" in df.columns

    meta_by_id = {}
    for _, row in df.iterrows():
        sid = sample_id_from_meta_filename(row["filename"])
        meta_by_id[str(sid)] = row

    y_str = {}
    g_frog = {}
    uhm_sample = {}
    missing = []

    for p in npz_files:
        sid = sample_id_from_npz_name(p.name)
        if str(sid) not in meta_by_id:
            missing.append(p.name)
            continue

        r = meta_by_id[str(sid)]
        uhm = str(r["UHM_sample"])
        trt = str(r["treatment"])
        frog = frog_from_uhm_sample(uhm)

        y_str[p.name] = trt
        g_frog[p.name] = frog
        uhm_sample[p.name] = uhm

    if len(missing) > 0:
        print(f"[WARN] {len(missing)} npz files not matched. Example:", missing[:5])

    return y_str, g_frog, uhm_sample


# =========================
# C) build eval loader
# =========================
def build_eval_loader(npz_dir, seed=0, window_size=500, stride=250):
    out_dir = Path(npz_dir)
    npz_files = sorted(out_dir.glob("*.npz"))
    print("Total npz:", len(npz_files))

    slicer = WindowSlicer(window_size=window_size, stride=stride, jitter_max=0)

    eval_ds = MassSpecWindowDataset(
        npz_files=npz_files,
        window_slicer=slicer,
        augment=None,
        n_windows_target=None,
        seed=seed,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=256,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_window_level_with_name,
        drop_last=False,
    )
    return npz_files, eval_loader


# =========================
# D) load encoder
# =========================
def load_encoder_from_ckpt(
    ckpt_path: str,
    device: str,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt)

    load_sd, used_prefix = strip_prefix_any(
        state,
        prefixes=["enc_with_head.encoder.", "encoder.", "model.encoder."]
    )
    if load_sd is None:
        raise KeyError(f"Cannot find encoder prefix. Example keys: {list(state.keys())[:5]}")

    enc = MassSpecWindowContrastEncoder(
        embed_dim=embed_dim,
        patch_size=patch_size,
        stride=encoder_stride,
        num_heads=num_heads,
    ).to(device)

    missing, unexpected = enc.load_state_dict(load_sd, strict=False)
    enc.eval()

    print("loaded:", ckpt_path)
    print("used_prefix:", used_prefix, "| missing:", len(missing), "| unexpected:", len(unexpected))
    return enc


# =========================
# E) extract E_file
# =========================
@torch.no_grad()
def extract_E_file_from_encoder(encoder, loader, device, y_file_str_map):
    Z_sum = defaultdict(lambda: None)
    Z_cnt = defaultdict(int)
    y_str = {}

    encoder.eval()
    encoder.to(device)

    for batch in loader:
        signal = batch["signal"].to(device)
        rt     = batch["rt"].to(device)
        names  = list(batch["file_name"])

        z = encoder(signal, rt)
        z = z.detach().cpu().numpy().astype(np.float64)

        for zi, fn in zip(z, names):
            fn = str(fn)
            if fn not in y_file_str_map:
                continue
            if Z_sum[fn] is None:
                Z_sum[fn] = zi.copy()
            else:
                Z_sum[fn] += zi
            Z_cnt[fn] += 1
            y_str[fn] = y_file_str_map[fn]

    file_names = sorted(Z_sum.keys())
    E_file = np.stack([Z_sum[n] / max(1, Z_cnt[n]) for n in file_names], axis=0)
    y_file_str = np.array([y_str[n] for n in file_names], dtype=object)
    g_file = np.array([frog_from_npz_name(n) for n in file_names], dtype=str)

    return E_file, y_file_str, g_file, file_names


# =========================
# F) task-specific LOGO frog-level metrics
# =========================
def aggregate_by_group(values, groups, mode="mean"):
    buckets = defaultdict(list)
    for v, g in zip(values, groups):
        buckets[g].append(float(v))

    group_ids = list(buckets.keys())

    if mode == "mean":
        agg = np.array([np.mean(buckets[g]) for g in group_ids], dtype=float)
    elif mode == "median":
        agg = np.array([np.median(buckets[g]) for g in group_ids], dtype=float)
    else:
        raise ValueError(mode)

    return agg, np.array(group_ids, dtype=object)


def logo_binary_metrics_for_task(
    E,
    y_str,
    groups_frog,
    pos_class,
    neg_class="control",
    C=10.0,
    threshold=0.5,
    frog_agg="mean",
):
    E = np.asarray(E)
    y_str = np.asarray(y_str, dtype=object)
    g = np.asarray(groups_frog, dtype=object)

    keep = np.isin(y_str, [pos_class, neg_class])
    E2, y2_str, g2 = E[keep], y_str[keep], g[keep]

    if E2.shape[0] == 0:
        raise ValueError(f"No samples for {pos_class} vs {neg_class}")

    y2 = (y2_str == pos_class).astype(int)
    logo = LeaveOneGroupOut()

    P_file_all, Y_file_all, G_file_all = [], [], []
    n_skip = 0

    for tr, te in logo.split(E2, y2, g2):
        if len(np.unique(y2[tr])) < 2:
            n_skip += 1
            continue

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=C,
                max_iter=5000,
                class_weight="balanced",
                solver="lbfgs",
            ),
        )
        clf.fit(E2[tr], y2[tr])

        p = clf.predict_proba(E2[te])[:, 1]
        P_file_all.append(p)
        Y_file_all.append(y2[te])
        G_file_all.append(g2[te])

    if len(P_file_all) == 0:
        raise ValueError(f"All folds skipped. skipped={n_skip}")

    P_file = np.concatenate(P_file_all)
    Y_file = np.concatenate(Y_file_all)
    G_file = np.concatenate(G_file_all)

    P_frog, frogs = aggregate_by_group(P_file, G_file, mode=frog_agg)
    Y_frog_mean, _ = aggregate_by_group(Y_file, G_file, mode="mean")
    Y_frog = (Y_frog_mean >= 0.5).astype(int)

    pred_frog = (P_frog >= threshold).astype(int)
    frog_acc = accuracy_score(Y_frog, pred_frog)

    if len(np.unique(Y_frog)) < 2:
        frog_auroc = np.nan
        frog_auprc = np.nan
    else:
        frog_auroc = roc_auc_score(Y_frog, P_frog)
        frog_auprc = average_precision_score(Y_frog, P_frog)

    return {
        "AUROC": float(frog_auroc),
        "AUPRC": float(frog_auprc),
        "ACC": float(frog_acc),
        "n_frogs": int(len(Y_frog)),
        "n_pos_frogs": int(Y_frog.sum()),
        "n_neg_frogs": int((1 - Y_frog).sum()),
        "skipped_folds": int(n_skip),
    }


# =========================
# G) evaluate one checkpoint for one task
# =========================
def eval_one_ckpt_one_task(
    ckpt_path,
    eval_loader,
    y_file_str_map,
    device,
    pos_class,
    neg_class="control",
    C=10.0,
    frog_agg="mean",
    encoder_kwargs=None,
):
    encoder_kwargs = {} if encoder_kwargs is None else dict(encoder_kwargs)

    encoder = load_encoder_from_ckpt(
        ckpt_path,
        device=device,
        **encoder_kwargs,
    )

    E_file, y_file_str, g_file, file_names = extract_E_file_from_encoder(
        encoder, eval_loader, device, y_file_str_map
    )

    r = logo_binary_metrics_for_task(
        E=E_file,
        y_str=y_file_str,
        groups_frog=g_file,
        pos_class=pos_class,
        neg_class=neg_class,
        C=C,
        frog_agg=frog_agg,
    )

    return r


# =========================
# H) evaluate one checkpoint for both tasks
# =========================
def eval_one_ckpt_two_tasks(
    ckpt_path,
    eval_loader,
    y_file_str_map,
    device,
    tasks=(("STP1710.7", "control"),),
    C=10.0,
    frog_agg="mean",
    encoder_kwargs=None,
):
    row = {
        "ckpt": Path(ckpt_path).name,
        "ckpt_path": str(ckpt_path),
        "step": parse_step_from_ckpt_name(Path(ckpt_path).name),
        "exp_name": parse_exp_name_from_ckpt_path(ckpt_path),
    }
    row["seed"] = parse_seed_from_exp_name(row["exp_name"])
    row["config"] = parse_config_from_exp_name(row["exp_name"])

    for pos_class, neg_class in tasks:
        task_key = f"{pos_class}_vs_{neg_class}"
        m = eval_one_ckpt_one_task(
            ckpt_path=ckpt_path,
            eval_loader=eval_loader,
            y_file_str_map=y_file_str_map,
            device=device,
            pos_class=pos_class,
            neg_class=neg_class,
            C=C,
            frog_agg=frog_agg,
            encoder_kwargs=encoder_kwargs,
        )
        row[f"AUROC_{task_key}"] = m["AUROC"]
        row[f"AUPRC_{task_key}"] = m["AUPRC"]
        row[f"ACC_{task_key}"] = m["ACC"]
        row[f"n_frogs_{task_key}"] = m["n_frogs"]
        row[f"n_pos_frogs_{task_key}"] = m["n_pos_frogs"]
        row[f"n_neg_frogs_{task_key}"] = m["n_neg_frogs"]
        row[f"skipped_folds_{task_key}"] = m["skipped_folds"]

    task_keys = [f"{pos}_vs_{neg}" for pos, neg in tasks]
    row["mean_AUROC_ctrl"] = float(np.nanmean([row[f"AUROC_{k}"] for k in task_keys]))
    row["mean_AUPRC_ctrl"] = float(np.nanmean([row[f"AUPRC_{k}"] for k in task_keys]))
    row["mean_ACC_ctrl"]   = float(np.nanmean([row[f"ACC_{k}"]   for k in task_keys]))

    return row


# =========================
# I) evaluate all saved checkpoints from df_runs
# =========================
def evaluate_all_saved_ckpts(
    df_runs,
    eval_loader,
    y_file_str_map,
    device,
    tasks=(("STP1710.7", "control"),),
    C=10.0,
    frog_agg="mean",
    encoder_kwargs=None,
):
    rows = []

    for _, run_row in df_runs.iterrows():
        ckpt_dir = Path(run_row["ckpt_dir"])
        ckpt_paths = sorted(ckpt_dir.rglob("*.ckpt"))
        print(f"\n=== {run_row['exp_name']} | seed={run_row['seed']} | n_ckpts={len(ckpt_paths)} ===")

        for ckpt_path in ckpt_paths:
            try:
                row = eval_one_ckpt_two_tasks(
                    ckpt_path=str(ckpt_path),
                    eval_loader=eval_loader,
                    y_file_str_map=y_file_str_map,
                    device=device,
                    tasks=tasks,
                    C=C,
                    frog_agg=frog_agg,
                    encoder_kwargs=encoder_kwargs,
                )

                if "config" in run_row.index and pd.notna(run_row["config"]):
                    row["grid_config"] = run_row["config"]
                if "grid_params" in run_row.index:
                    row["grid_params"] = json.dumps(run_row["grid_params"], sort_keys=True) \
                        if isinstance(run_row["grid_params"], dict) else str(run_row["grid_params"])

                rows.append(row)

                print(
                    f"OK | {ckpt_path.name} | "
                    f"1710 AUROC={row['AUROC_STP1710.7_vs_control']:.4f} | "
                    f"1710 AUROC={row['AUROC_STP1710.7_vs_control']:.4f} | "
                    f"mean_AUROC_ctrl={row['mean_AUROC_ctrl']:.4f}"
                )
            except Exception as e:
                print("FAIL:", ckpt_path.name, "->", repr(e))

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    sort_cols = [
        "config", "seed", "step",
        "AUROC_STP1710.7_vs_control",
    ]
    sort_cols = [c for c in sort_cols if c in df.columns]
    df = df.sort_values(sort_cols).reset_index(drop=True)
    return df


# =========================
# J) optional inspection tables (no final model decision)
# =========================
def make_inspection_tables(df_all):
    if len(df_all) == 0:
        return {
            "all_results": df_all.copy(),
            "best_1710_per_config_seed": pd.DataFrame(),
            "best_1710_per_config_seed": pd.DataFrame(),
            "best_mean_per_config_seed": pd.DataFrame(),
        }

    group_cols = ["config", "seed"]
    if "grid_config" in df_all.columns:
        group_cols = ["grid_config", "seed"]

    best_1710 = df_all.loc[
        df_all.groupby(group_cols)["AUROC_STP1710.7_vs_control"].idxmax()
    ].sort_values(group_cols + ["AUROC_STP1710.7_vs_control"], ascending=[True, True, False])

    return {
        "all_results": df_all.copy(),
        "best_1710_per_config_seed": best_1710.reset_index(drop=True),
    }



META_CSV=META_PATH
# NPZ_DIR = "data/processed/mzML_npz_45"
# META_CSV = "data/processed/metadata_with_frog.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVAL_SEED = 0











npz_files, eval_loader = build_eval_loader(NPZ_DIR, seed=EVAL_SEED)
y_file_str_map, g_frog_map, uhm_sample_map = build_npz_to_label_maps(npz_files, META_CSV)

print("DEVICE:", DEVICE)
print("Matched files:", len(y_file_str_map))

In [ ]:

# =========================
# Run evaluation across all saved checkpoints + sweep LogisticRegression C
# =========================
TASKS = (
    ("STP1710.7", "control"),
)

C_GRID = [40]

def evaluate_all_saved_ckpts_c_sweep(
    df_runs,
    eval_loader,
    y_file_str_map,
    device,
    tasks,
    c_grid,
    frog_agg="mean",
    encoder_kwargs=None,
):
    rows = []

    for _, run_row in df_runs.iterrows():
        ckpt_dir = Path(run_row["ckpt_dir"])
        ckpt_paths = sorted(ckpt_dir.rglob("*.ckpt"))
        print(f"\n=== {run_row['exp_name']} | seed={run_row['seed']} | n_ckpts={len(ckpt_paths)} ===")

        for ckpt_path in ckpt_paths:
            for C in c_grid:
                try:
                    row = eval_one_ckpt_two_tasks(
                        ckpt_path=str(ckpt_path),
                        eval_loader=eval_loader,
                        y_file_str_map=y_file_str_map,
                        device=device,
                        tasks=tasks,
                        C=C,
                        frog_agg=frog_agg,
                        encoder_kwargs=encoder_kwargs,
                    )

                    row["C"] = float(C)

                    if "config" in run_row.index and pd.notna(run_row["config"]):
                        row["grid_config"] = run_row["config"]
                    if "grid_params" in run_row.index:
                        row["grid_params"] = (
                            json.dumps(run_row["grid_params"], sort_keys=True)
                            if isinstance(run_row["grid_params"], dict)
                            else str(run_row["grid_params"])
                        )

                    rows.append(row)

                    msg = (
                        f"OK | {ckpt_path.name} | C={C:g} | "
                        f"1710 AUROC={row['AUROC_STP1710.7_vs_control']:.4f} | "
                        f"AUPRC={row['AUPRC_STP1710.7_vs_control']:.4f} | "
                        f"ACC={row['ACC_STP1710.7_vs_control']:.4f}"
                    )
                    print(msg)

                except Exception as e:
                    print(f"FAIL: {ckpt_path.name} | C={C:g} -> {repr(e)}")

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    sort_cols = [
        "config", "seed", "step", "C", "AUROC_STP1710.7_vs_control"
    ]
    sort_cols = [c for c in sort_cols if c in df.columns]
    ascending = [True, True, True, True, False][:len(sort_cols)]
    df = df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
    return df


def make_task1710_c_sweep_tables(df_all):
    if len(df_all) == 0:
        return {
            "all_results": df_all.copy(),
            "best_per_config_seed": pd.DataFrame(),
            "best_per_config": pd.DataFrame(),
            "best_overall": pd.DataFrame(),
        }

    group_cols = ["grid_config", "seed"] if "grid_config" in df_all.columns else ["config", "seed"]

    metric_col = "AUROC_STP1710.7_vs_control"
    sort_cols = group_cols + [metric_col, "AUPRC_STP1710.7_vs_control", "ACC_STP1710.7_vs_control"]
    ascending = [True] * len(group_cols) + [False, False, False]

    best_per_config_seed = df_all.loc[
        df_all.groupby(group_cols)[metric_col].idxmax()
    ].sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

    config_col = "grid_config" if "grid_config" in best_per_config_seed.columns else "config"
    summary = (
        best_per_config_seed
        .groupby(config_col, as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            mean_AUROC_STP1710_7_vs_control=(metric_col, "mean"),
            std_AUROC_STP1710_7_vs_control=(metric_col, "std"),
            mean_AUPRC_STP1710_7_vs_control=("AUPRC_STP1710.7_vs_control", "mean"),
            mean_ACC_STP1710_7_vs_control=("ACC_STP1710.7_vs_control", "mean"),
        )
        .sort_values(
            ["mean_AUROC_STP1710_7_vs_control", "mean_AUPRC_STP1710_7_vs_control"],
            ascending=[False, False]
        )
        .reset_index(drop=True)
    )

    best_overall = best_per_config_seed.sort_values(
        [metric_col, "AUPRC_STP1710.7_vs_control", "ACC_STP1710.7_vs_control"],
        ascending=[False, False, False]
    ).head(1).reset_index(drop=True)

    return {
        "all_results": df_all.copy(),
        "best_per_config_seed": best_per_config_seed,
        "best_per_config": summary,
        "best_overall": best_overall,
    }


ALL_CKPT_RESULTS_CSWEEP = evaluate_all_saved_ckpts_c_sweep(
    df_runs=df_runs_grid,
    eval_loader=eval_loader,
    y_file_str_map=y_file_str_map,
    device=DEVICE,
    tasks=TASKS,
    c_grid=C_GRID,
    frog_agg="mean",
    encoder_kwargs=dict(
        embed_dim=64,
        patch_size=50,
        encoder_stride=50,
        num_heads=4,
    ),
)

display(ALL_CKPT_RESULTS_CSWEEP.head())
print("Total evaluated checkpoint x C rows:", len(ALL_CKPT_RESULTS_CSWEEP))

#RESULTS_DIR = Path("./task1710_ckpt_eval_results/log5")

RESULTS_DIR = LOG_ROOT
#RESULTS_DIR=PROJECT_ROOT /"results"/"runs"/"training"/"Task1710"/"logs"/"logs8"/"C30"



RESULTS_DIR.mkdir(parents=True, exist_ok=True)

all_csv = RESULTS_DIR / "all_ckpt_metrics__STP1710.7_vs_control__c_sweep.csv"
ALL_CKPT_RESULTS_CSWEEP.to_csv(all_csv, index=False)
print("Saved:", all_csv)

TABLES = make_task1710_c_sweep_tables(ALL_CKPT_RESULTS_CSWEEP)

best_per_config_seed = TABLES["best_per_config_seed"]
best_per_config = TABLES["best_per_config"]
best_overall = TABLES["best_overall"]

best_seed_csv = RESULTS_DIR / "best_per_config_seed__STP1710.7_vs_control.csv"
best_cfg_csv = RESULTS_DIR / "best_per_config__STP1710.7_vs_control.csv"
best_overall_csv = RESULTS_DIR / "best_overall__STP1710.7_vs_control.csv"

best_per_config_seed.to_csv(best_seed_csv, index=False)
best_per_config.to_csv(best_cfg_csv, index=False)
best_overall.to_csv(best_overall_csv, index=False)

print("Saved:", best_seed_csv)
print("Saved:", best_cfg_csv)
print("Saved:", best_overall_csv)

print("\n===== Full results (first rows) =====")
display(
    ALL_CKPT_RESULTS_CSWEEP[
        [
            "config", "seed", "step", "C", "ckpt_path",
            "AUROC_STP1710.7_vs_control",
            "AUPRC_STP1710.7_vs_control",
            "ACC_STP1710.7_vs_control",
        ]
    ].head(20)
)

print("\n===== Best checkpoint + best C within each config-seed for 1710 AUROC =====")
display(
    best_per_config_seed[
        [
            "config", "seed", "step", "C", "ckpt_path",
            "AUROC_STP1710.7_vs_control",
            "AUPRC_STP1710.7_vs_control",
            "ACC_STP1710.7_vs_control",
        ]
    ]
)

print("\n===== Mean performance over seeds (best checkpoint + best C picked inside each seed) =====")
display(best_per_config)

print("\n===== Best overall row =====")
display(best_overall)
